# AMEX Explainability Report Generator

Generates one Word report per customer with prediction, SHAP, LIME, model-based counterfactuals, OpenAI narrative, charts, and technical tables.

In [ ]:
!pip -q install openai xgboost shap lime python-docx openpyxl joblib scikit-learn pandas numpy matplotlib

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.7/275.7 kB 6.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 21.1 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount("/content/drive/")

from pathlib import Path
from getpass import getpass
import os

MODEL_FOLDER = Path("/content/drive/AMEX_XGBoost_23_Features_Final")
CUSTOMER_FILE = Path("/content/Excel Template.xlsx")
REFERENCE_FILE = Path("/content/AMEX_Filtered_100K_Finaldataset.csv")

OUTPUT_FOLDER = Path("/content/credit_risk_reports_final")
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)

ID_COLUMN = "customer_ID"
TARGET_COLUMN = "target"
CLASSIFICATION_THRESHOLD = 0.50
OPENAI_MODEL = "gpt-5.6-luna"
TOP_SHAP_FEATURES = 10
TOP_LIME_FEATURES = 10
MAX_COUNTERFACTUAL_DRIVERS = 5
RANDOM_STATE = 42

os.environ["OPENAI_API_KEY"] = getpass("Key")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import re, json, hashlib, zipfile, warnings, joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap

from lime.lime_tabular import LimeTabularExplainer
from docx import Document
from docx.shared import Inches
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.enum.table import WD_TABLE_ALIGNMENT
from openai import OpenAI

warnings.filterwarnings("ignore")
client = OpenAI()

def find_model_file(folder):
    preferred = list(folder.glob("*PowerBI*.pkl"))
    candidates = preferred or list(folder.glob("*.pkl"))
    if not candidates:
        raise FileNotFoundError(f"No .pkl file found inside {folder}")
    print("Using model:", candidates[0].name)
    return candidates[0]

def load_bundle(path):
    loaded = joblib.load(path)
    if isinstance(loaded, dict):
        model = None
        for key in ["model","xgb_model","classifier","estimator","best_model","pipeline"]:
            if loaded.get(key) is not None:
                model = loaded.get(key)
                break
        features = None
        for key in ["feature_names","features","selected_features","model_features"]:
            if loaded.get(key) is not None:
                features = loaded.get(key)
                break
        holdout_auc = loaded.get("holdout_auc")
        best_params = loaded.get("best_params", {})
    else:
        model = loaded
        features = getattr(model, "feature_names_in_", None)
        holdout_auc = None
        best_params = {}

    if model is None:
        raise ValueError("Model not found in bundle.")

    if features is None:
        features = getattr(model, "feature_names_in_", None)

    if features is None and hasattr(model, "get_booster"):
        features = model.get_booster().feature_names

    if features is None:
        raise ValueError("Feature names not found in bundle.")

    return {
        "model": model,
        "feature_names": list(features),
        "holdout_auc": holdout_auc,
        "best_params": best_params
    }

def to_float(value):
    if value is None:
        return np.nan
    if isinstance(value, (int,float,np.integer,np.floating)):
        return float(value)
    if isinstance(value, (list,tuple,np.ndarray)):
        arr = np.asarray(value).reshape(-1)
        return to_float(arr[0]) if len(arr) else np.nan

    text = str(value).strip()
    if text.lower() in {"","nan","none","null","na","n/a"}:
        return np.nan

    while ((text.startswith("[") and text.endswith("]")) or
           (text.startswith("(") and text.endswith(")"))):
        text = text[1:-1].strip()

    if "," in text:
        text = text.split(",")[0].strip()

    try:
        return float(text)
    except ValueError:
        match = re.search(r"[-+]?(?:\d*\.\d+|\d+\.?)(?:[eE][-+]?\d+)?", text)
        return float(match.group()) if match else np.nan

def clean_features(frame, features, medians=None):
    missing = [f for f in features if f not in frame.columns]
    if missing:
        raise ValueError(f"Missing features: {missing}")

    out = frame[features].copy()
    for feature in features:
        out[feature] = out[feature].map(to_float)

    out = out.replace([np.inf,-np.inf], np.nan)
    if medians is not None:
        out = out.fillna(medians)
    return out.astype(float)

bundle = load_bundle(find_model_file(MODEL_FOLDER))
model = bundle["model"]
FEATURES = bundle["feature_names"]

customers_df = pd.read_excel(CUSTOMER_FILE)
reference_df = pd.read_csv(REFERENCE_FILE)

reference_x = clean_features(reference_df, FEATURES)
reference_medians = reference_x.median().fillna(0.0)
reference_x = reference_x.fillna(reference_medians)

customers_x = clean_features(customers_df, FEATURES, reference_medians)
customers_df.loc[:, FEATURES] = customers_x

print("Model:", type(model))
print("Features:", len(FEATURES))
print("Customers:", len(customers_df))

Using model: AMEX_XGBoost_23_Features_PowerBI.pkl
Model: <class 'xgboost.sklearn.XGBClassifier'>
Features: 23
Customers: 2


In [ ]:
FEATURE_DICTIONARY = {
    "P_2_last":"payment behaviour, latest available value",
    "R_1_last":"risk indicator behaviour, latest available value",
    "D_48_last":"recent delinquency or repayment-delay behaviour",
    "B_22_min_6m":"account balance behaviour, lowest recent value",
    "B_4_trend_slope":"direction of account balance movement",
    "B_3_std_6m_y":"account balance volatility over six months",
    "B_3_std_12m_y":"account balance volatility over twelve months",
    "R_1_mean_12m":"risk indicator behaviour, average recent value",
    "S_8_mean_6m":"spending behaviour, average recent value",
    "S_11_min_12m":"spending behaviour, lowest recent value",
    "D_42_mean_6m":"recent delinquency behaviour, average recent value",
    "D_43_last":"recent delinquency behaviour, latest available value",
    "P_to_B_last_ratio_3m":"recent payment-to-balance relationship"
}

ACTIONABLE_FEATURES = [
    f for f in [
        "P_2_last","R_1_last","D_48_last","B_4_trend_slope",
        "B_3_std_6m_y","B_3_std_12m_y","R_1_mean_12m",
        "S_8_mean_6m","S_11_min_12m","D_42_mean_6m",
        "D_43_last","P_to_B_last_ratio_3m"
    ] if f in FEATURES
]

def plain_name(feature):
    return FEATURE_DICTIONARY.get(feature, feature.replace("_"," "))

def predict_pd(frame):
    x = clean_features(frame, FEATURES, reference_medians)
    proba = np.asarray(model.predict_proba(x))
    return proba[:,1].astype(float)

def pd_to_score(pd_value):
    return int(round(np.clip(900 - 600*float(pd_value), 300, 900)))

def risk_band(pd_value):
    if pd_value >= .75: return "Very High"
    if pd_value >= .50: return "High"
    if pd_value >= .25: return "Moderate"
    if pd_value >= .10: return "Low"
    return "Very Low"

customers_df["Predicted_Default_Probability"] = predict_pd(customers_df)
customers_df["Predicted_Default_Flag"] = (
    customers_df["Predicted_Default_Probability"] >= CLASSIFICATION_THRESHOLD
).astype(int)
customers_df["Model_Credit_Score"] = customers_df["Predicted_Default_Probability"].map(pd_to_score)
customers_df["Risk_Band"] = customers_df["Predicted_Default_Probability"].map(risk_band)

background = reference_x.sample(n=min(200,len(reference_x)), random_state=RANDOM_STATE)

def shap_predict(values):
    return predict_pd(pd.DataFrame(values, columns=FEATURES))

try:
    shap_explainer = shap.TreeExplainer(model)
    SHAP_MODE = "tree"
except Exception:
    shap_explainer = shap.Explainer(
        shap_predict,
        shap.maskers.Independent(background),
        algorithm="permutation",
        feature_names=FEATURES
    )
    SHAP_MODE = "permutation"

def calculate_shap(customer_row):
    customer_x = clean_features(pd.DataFrame([customer_row]), FEATURES, reference_medians)

    if SHAP_MODE == "tree":
        exp = shap_explainer(customer_x, check_additivity=False)
    else:
        exp = shap_explainer(customer_x, max_evals=max(2*len(FEATURES)+1,100))

    values = np.asarray(exp.values)
    if values.ndim == 2:
        values = values[0]
    elif values.ndim == 3:
        values = values[0,:, -1]
    else:
        raise ValueError(f"Unexpected SHAP shape: {values.shape}")

    table = pd.DataFrame({
        "feature": FEATURES,
        "plain_name": [plain_name(f) for f in FEATURES],
        "value": customer_x.iloc[0].values,
        "shap_value": values.astype(float)
    })
    table["direction"] = np.where(
        table["shap_value"] > 0, "increased risk",
        np.where(table["shap_value"] < 0, "reduced risk", "neutral")
    )
    table["absolute_impact"] = table["shap_value"].abs()
    return table.sort_values("absolute_impact", ascending=False).reset_index(drop=True)

lime_explainer = LimeTabularExplainer(
    training_data=reference_x.values,
    feature_names=FEATURES,
    class_names=["Non-default","Default"],
    mode="classification",
    discretize_continuous=True,
    random_state=RANDOM_STATE
)

def lime_predict(values):
    frame = pd.DataFrame(values, columns=FEATURES)
    x = clean_features(frame, FEATURES, reference_medians)
    return np.asarray(model.predict_proba(x))

def calculate_lime(customer_row):
    customer_x = clean_features(pd.DataFrame([customer_row]), FEATURES, reference_medians)
    exp = lime_explainer.explain_instance(
        customer_x.iloc[0].to_numpy(dtype=float),
        lime_predict,
        num_features=min(TOP_LIME_FEATURES,len(FEATURES)),
        labels=(1,)
    )
    return pd.DataFrame([
        {
            "lime_rule": rule,
            "lime_weight": float(weight),
            "direction": "increased risk" if weight > 0 else "reduced risk"
        }
        for rule, weight in exp.as_list(label=1)
    ])

In [ ]:

if TARGET_COLUMN in reference_df.columns:
    mask = pd.to_numeric(reference_df[TARGET_COLUMN], errors="coerce").fillna(0).astype(int) == 0
    low_risk_reference = reference_x.loc[mask]
else:
    ref_pd = predict_pd(reference_df)
    low_risk_reference = reference_x.loc[ref_pd <= np.quantile(ref_pd,0.30)]

low_risk_medians = low_risk_reference.median()

def generate_counterfactuals(customer_row, shap_table):
    current_pd = float(customer_row["Predicted_Default_Probability"])
    original = clean_features(pd.DataFrame([customer_row]), FEATURES, reference_medians).iloc[0]

    drivers = shap_table.loc[
        (shap_table["shap_value"] > 0) &
        shap_table["feature"].isin(ACTIONABLE_FEATURES)
    ].head(MAX_COUNTERFACTUAL_DRIVERS)

    rows = []
    cumulative = original.copy()
    changed = []

    for rank, driver in enumerate(drivers.itertuples(), start=1):
        feature = driver.feature
        target_value = float(low_risk_medians[feature])

        single = original.copy()
        single[feature] = target_value
        single_pd = float(predict_pd(pd.DataFrame([single], columns=FEATURES))[0])

        rows.append({
            "scenario": f"Adjust {plain_name(feature)} toward lower-risk range",
            "changed_features": feature,
            "pd_before": current_pd,
            "pd_after_change": single_pd,
            "pd_reduction": current_pd-single_pd,
            "new_credit_score": pd_to_score(single_pd),
            "crosses_threshold": single_pd < CLASSIFICATION_THRESHOLD
        })

        cumulative[feature] = target_value
        changed.append(feature)
        cumulative_pd = float(predict_pd(pd.DataFrame([cumulative], columns=FEATURES))[0])

        rows.append({
            "scenario": f"Cumulative improvement through top {rank} driver(s)",
            "changed_features": ", ".join(changed),
            "pd_before": current_pd,
            "pd_after_change": cumulative_pd,
            "pd_reduction": current_pd-cumulative_pd,
            "new_credit_score": pd_to_score(cumulative_pd),
            "crosses_threshold": cumulative_pd < CLASSIFICATION_THRESHOLD
        })

    return pd.DataFrame(rows).sort_values("pd_after_change").reset_index(drop=True) if rows else pd.DataFrame()

def create_shap_chart(table, title, path):
    plot_df = table.head(TOP_SHAP_FEATURES).sort_values("shap_value")
    colors = ["firebrick" if v > 0 else "seagreen" for v in plot_df["shap_value"]]
    plt.figure(figsize=(9,6))
    plt.barh(plot_df["plain_name"], plot_df["shap_value"], color=colors)
    plt.axvline(0,color="black",linewidth=1)
    plt.title(title)
    plt.xlabel("Impact on predicted default risk")
    plt.tight_layout()
    plt.savefig(path,dpi=180,bbox_inches="tight")
    plt.close()

def create_lime_chart(table, title, path):
    plot_df = table.sort_values("lime_weight")
    colors = ["firebrick" if v > 0 else "seagreen" for v in plot_df["lime_weight"]]
    plt.figure(figsize=(9,6))
    plt.barh(plot_df["lime_rule"], plot_df["lime_weight"], color=colors)
    plt.axvline(0,color="black",linewidth=1)
    plt.title(title)
    plt.xlabel("Local contribution to default class")
    plt.tight_layout()
    plt.savefig(path,dpi=180,bbox_inches="tight")
    plt.close()

def create_cf_chart(table, current_pd, title, path):
    if table.empty:
        return
    plot_df = table.head(8).copy()
    plot_df["id"] = [f"S{i}" for i in range(1,len(plot_df)+1)]
    plt.figure(figsize=(9,5))
    plt.barh(plot_df["id"], plot_df["pd_after_change"])
    plt.axvline(current_pd,linestyle="--",color="firebrick",label="Current PD")
    plt.axvline(CLASSIFICATION_THRESHOLD,linestyle=":",color="black",label="Threshold")
    plt.title(title)
    plt.xlabel("Predicted default probability")
    plt.legend()
    plt.tight_layout()
    plt.savefig(path,dpi=180,bbox_inches="tight")
    plt.close()

In [ ]:
SYSTEM_INSTRUCTIONS = '''
You are a senior credit-risk model governance analyst.
Write a professional customer-level report using only supplied evidence.

Do not invent facts.
SHAP and LIME explain model behaviour, not causality.
Counterfactuals are hypothetical model simulations.
Do not promise approval or prevention of default.
Use supportive, neutral banking language.

Return exactly these headings:
1. Risk summary
2. Main reasons increasing risk
3. Main reasons reducing risk
4. SHAP and LIME consistency assessment
5. What could improve the profile
6. Counterfactual interpretation
7. Recommended internal follow-up
8. Plain-language explanation
9. Governance limitations
'''

def pseudonymize(value):
    return hashlib.sha256(str(value).encode()).hexdigest()[:12]

def records(frame, n=10):
    if frame is None or frame.empty:
        return []
    return json.loads(
        frame.head(n).replace([np.inf,-np.inf],np.nan).to_json(orient="records")
    )

def generate_narrative(customer_row, shap_table, lime_table, cf_table):
    pd_value = float(customer_row["Predicted_Default_Probability"])
    evidence = {
        "customer_reference": pseudonymize(customer_row[ID_COLUMN]),
        "prediction": {
            "predicted_default_probability": pd_value,
            "classification_threshold": CLASSIFICATION_THRESHOLD,
            "predicted_class": "Potential Defaulter" if pd_value >= CLASSIFICATION_THRESHOLD else "Potential Non-defaulter",
            "credit_score": int(customer_row["Model_Credit_Score"]),
            "risk_band": customer_row["Risk_Band"],
            "holdout_auc": bundle["holdout_auc"]
        },
        "risk_increasing_factors": records(
            shap_table.loc[shap_table["shap_value"] > 0,
                           ["feature","plain_name","value","shap_value"]], 5
        ),
        "risk_reducing_factors": records(
            shap_table.loc[shap_table["shap_value"] < 0,
                           ["feature","plain_name","value","shap_value"]], 5
        ),
        "lime_rules": records(lime_table,10),
        "counterfactual_scenarios": records(cf_table,8)
    }

    response = client.responses.create(
        model=OPENAI_MODEL,
        instructions=SYSTEM_INSTRUCTIONS,
        input=json.dumps(evidence,indent=2)
    )
    return response.output_text.strip()

def add_dataframe(document, frame, columns=None, n=10):
    if frame is None or frame.empty:
        document.add_paragraph("No results were available.")
        return

    table_df = frame.copy()
    if columns:
        table_df = table_df[[c for c in columns if c in table_df.columns]]
    table_df = table_df.head(n)

    table = document.add_table(rows=1, cols=len(table_df.columns))
    table.style = "Table Grid"
    table.alignment = WD_TABLE_ALIGNMENT.CENTER

    for i,col in enumerate(table_df.columns):
        table.rows[0].cells[i].text = str(col)

    for _,row in table_df.iterrows():
        cells = table.add_row().cells
        for i,value in enumerate(row):
            if isinstance(value,(float,np.floating)):
                cells[i].text = "" if pd.isna(value) else f"{value:.4f}"
            else:
                cells[i].text = "" if pd.isna(value) else str(value)

def add_text(document, narrative):
    for raw in narrative.splitlines():
        line = raw.strip()
        if not line:
            continue
        if re.match(r"^\d+\.\s+",line):
            document.add_heading(line,level=1)
        elif line.startswith(("- ","• ","* ")):
            document.add_paragraph(line[2:].strip(),style="List Bullet")
        else:
            document.add_paragraph(line)

def create_customer_report(customer_row):
    customer_id = str(customer_row[ID_COLUMN])
    ref = pseudonymize(customer_id)
    pd_value = float(customer_row["Predicted_Default_Probability"])

    shap_table = calculate_shap(customer_row)
    lime_table = calculate_lime(customer_row)
    cf_table = generate_counterfactuals(customer_row, shap_table)

    shap_path = OUTPUT_FOLDER/f"{ref}_shap.png"
    lime_path = OUTPUT_FOLDER/f"{ref}_lime.png"
    cf_path = OUTPUT_FOLDER/f"{ref}_counterfactual.png"

    create_shap_chart(shap_table,f"SHAP explanation – {ref}",shap_path)
    create_lime_chart(lime_table,f"LIME explanation – {ref}",lime_path)
    create_cf_chart(cf_table,pd_value,f"Counterfactual scenarios – {ref}",cf_path)

    narrative = generate_narrative(customer_row,shap_table,lime_table,cf_table)

    doc = Document()
    title = doc.add_heading("Explainable Credit Risk Report",level=0)
    title.alignment = WD_ALIGN_PARAGRAPH.CENTER

    segment = "Potential Defaulter" if pd_value >= CLASSIFICATION_THRESHOLD else "Potential Non-defaulter"
    summary = [
        ("Customer ID",customer_id),
        ("Segment",segment),
        ("Predicted default probability",f"{pd_value:.2%}"),
        ("Credit risk score (300–900)",str(customer_row["Model_Credit_Score"])),
        ("Risk band",customer_row["Risk_Band"]),
        ("Production threshold",f"{CLASSIFICATION_THRESHOLD:.2f}")
    ]

    table = doc.add_table(rows=len(summary),cols=2)
    table.style = "Table Grid"
    for i,(label,value) in enumerate(summary):
        table.rows[i].cells[0].text = label
        table.rows[i].cells[1].text = str(value)

    doc.add_paragraph()
    add_text(doc,narrative)

    doc.add_heading("SHAP explanation",level=1)
    doc.add_picture(str(shap_path),width=Inches(6.2))
    add_dataframe(doc,shap_table,
                  ["feature","plain_name","value","shap_value","direction"],12)

    doc.add_heading("LIME explanation",level=1)
    doc.add_picture(str(lime_path),width=Inches(6.2))
    add_dataframe(doc,lime_table,n=10)

    doc.add_heading("Counterfactual scenarios",level=1)
    doc.add_paragraph("These are model-based what-if simulations, not causal guarantees.")
    if cf_path.exists():
        doc.add_picture(str(cf_path),width=Inches(6.2))
    add_dataframe(doc,cf_table,
                  ["scenario","changed_features","pd_before","pd_after_change",
                   "pd_reduction","new_credit_score","crosses_threshold"],8)

    doc.add_heading("Technical traceability appendix",level=1)
    doc.add_paragraph(f"Model type: {type(model).__name__}")
    doc.add_paragraph(f"Classification threshold: {CLASSIFICATION_THRESHOLD:.2f}")
    doc.add_paragraph("Credit score = 900 − 600 × PD, clipped to 300–900.")
    if bundle["holdout_auc"] is not None:
        doc.add_paragraph(f"Model holdout AUC: {float(bundle['holdout_auc']):.4f}")

    doc.add_heading("Governance note",level=1)
    doc.add_paragraph(
        "SHAP and LIME explain model behaviour but do not establish causality. "
        "Counterfactuals must be validated against policy, fairness, feasibility "
        "and model-risk governance."
    )

    report_path = OUTPUT_FOLDER/f"credit_risk_report_{ref}.docx"
    doc.save(report_path)

    shap_table.to_csv(OUTPUT_FOLDER/f"{ref}_shap.csv",index=False)
    lime_table.to_csv(OUTPUT_FOLDER/f"{ref}_lime.csv",index=False)
    cf_table.to_csv(OUTPUT_FOLDER/f"{ref}_counterfactuals.csv",index=False)

    return report_path

In [ ]:
generated = []

for row_number, customer_row in customers_df.iterrows():
    try:
        report = create_customer_report(customer_row)
        generated.append(report)
        print("Created:",report.name)
    except Exception as error:
        print(f"Row {row_number} failed:",error)

zip_path = Path("/content/AMEX_Explainability_Reports.zip")

with zipfile.ZipFile(zip_path,"w",zipfile.ZIP_DEFLATED) as archive:
    for file_path in OUTPUT_FOLDER.rglob("*"):
        if file_path.is_file():
            archive.write(file_path,arcname=file_path.relative_to(OUTPUT_FOLDER))

print("Reports created:",len(generated))
print("ZIP:",zip_path)

from google.colab import files
files.download(str(zip_path))

Created: credit_risk_report_7457aaa7dbcf.docx
Created: credit_risk_report_f88e4d8e072c.docx
Reports created: 2
ZIP: /content/AMEX_Explainability_Reports.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>